# LangGraph Customer Support Agent with Snowflake AI Observability

This notebook demonstrates how to:

1. **Build a LangGraph agent** that uses Snowflake Cortex Search and Cortex Analyst as tools
2. **Instrument the agent with TruLens** (`TruGraph`) for full trace capture
3. **Run evaluations** using Snowflake AI Observability to measure answer relevance, context relevance, and groundedness

### Architecture

```
User Query
    │
    ▼
LangGraph ReAct Agent (ChatSnowflake LLM)
    ├── Tool: Cortex Search (unstructured case lookups)
    └── Tool: Cortex Analyst (structured metrics via semantic view)
    │
    ▼
TruGraph (auto-instrumentation) → Snowflake AI Observability
```

### Prerequisites

- Run `setup.sql` to create the `CUST_SUPPORT_DEMO.SUPPORT` schema, tables, semantic view, and Cortex Search service
- Snowflake account with Cortex features enabled
- `CORTEX_USER` database role and `CREATE EXTERNAL AGENT` / `CREATE TASK` / `EXECUTE TASK` privileges

In [1]:
%pip install --quiet -U langchain-core langchain-snowflake langgraph trulens-core trulens-providers-cortex trulens-connectors-snowflake trulens-apps-langgraph

Note: you may need to restart the kernel to use updated packages.


## 1. Snowflake Session Setup

In [2]:
import os

sorted(os.environ)[-30:]

['PYDEVD_IPYTHON_COMPATIBLE_DEBUGGING',
 'PYDEVD_USE_FRAME_EVAL',
 'PYTHONIOENCODING',
 'PYTHONUNBUFFERED',
 'PYTHON_FROZEN_MODULES',
 'SHELL',
 'SHLVL',
 'SNOWFLAKE_FORCE_GCP_USE_DOWNSCOPED_CREDENTIAL',
 'SNOWFLAKE_PASSWORD',
 'SNOWFLAKE_PAT',
 'SSH_AUTH_SOCK',
 'TERM',
 'TMPDIR',
 'USER',
 'VSCODE_CODE_CACHE_PATH',
 'VSCODE_CRASH_REPORTER_PROCESS_TYPE',
 'VSCODE_CWD',
 'VSCODE_ESM_ENTRYPOINT',
 'VSCODE_HANDLES_UNCAUGHT_ERRORS',
 'VSCODE_IPC_HOOK',
 'VSCODE_L10N_BUNDLE_LOCATION',
 'VSCODE_NLS_CONFIG',
 'VSCODE_PID',
 'XPC_FLAGS',
 'XPC_SERVICE_NAME',
 '_',
 '_CE_CONDA',
 '_CE_M',
 '__CFBundleIdentifier',
 '__CF_USER_TEXT_ENCODING']

In [4]:
import os

        # "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        # "user": os.getenv("SNOWFLAKE_USER"),
        # "password": os.getenv("SNOWFLAKE_PAT"), 
        # "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH"),
        # "database": DATABASE,
        # "schema": SCHEMA,
        # "role": ROLE


os.environ["SNOWFLAKE_ACCOUNT"] = "<your_account>"
os.environ["SNOWFLAKE_USER"] = "<your_username>"
# os.environ["SNOWFLAKE_PASSWORD"] = "<your_password>"
os.environ["SNOWFLAKE_WAREHOUSE"] = "COMPUTE_WH"
os.environ["SNOWFLAKE_DATABASE"] = "CUST_SUPPORT_DEMO"
os.environ["SNOWFLAKE_SCHEMA"] = "SUPPORT"

from langchain_snowflake import create_session_from_env

session = create_session_from_env()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

Connected: "CUST_SUPPORT_DEMO"."SUPPORT"


## 2. Define Snowflake Cortex Tools

In [5]:
import json
import textwrap
from langchain_core.tools import tool
from langchain_snowflake import SnowflakeCortexSearchRetriever

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

CORTEX_SEARCH_SERVICE = "CUST_SUPPORT_DEMO.SUPPORT.CASE_SEARCH_SERVICE"

retriever = SnowflakeCortexSearchRetriever(
    session=session,
    schema="CUST_SUPPORT_DEMO.SUPPORT",
    service_name=CORTEX_SEARCH_SERVICE,
    k=5,
    auto_format_for_rag=True,
    content_field="ISSUE_SUMMARY",
    join_separator="\n\n",
    fallback_to_page_content=True,
)


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def search_support_cases(query: str) -> str:
    """Search support case details, issue descriptions, and resolution summaries.
    Use for questions about specific issues, finding cases by topic or keyword,
    or looking up resolution steps for a type of problem."""
    docs = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)


print("Cortex Search tool defined.")
test_results = search_support_cases.invoke("authentication login failures")
print(textwrap.shorten(test_results, width=200, placeholder="..."))

Cortex Search tool defined.
Customer unable to log in after password reset. MFA token not being accepted despite correct entry. Resolution: Reset MFA configuration and re-enrolled the user. Cleared cached authentication...


In [6]:
import requests

SEMANTIC_VIEW = "CUST_SUPPORT_DEMO.SUPPORT.SUPPORT_ANALYTICS"


@tool
@instrument(span_type=SpanAttributes.SpanType.RETRIEVAL)
def query_support_analytics(question: str) -> str:
    """Query structured support metrics using natural language.
    Use for quantitative questions about case counts, CSAT scores, resolution times,
    escalation rates, agent performance comparisons, and trend analysis."""
    host = session.connection.host
    token = session.connection.rest.token
    # token = os.getenv("SNOWFLAKE_PAT")

    resp = requests.post(
        url=f"https://{host}/api/v2/cortex/analyst/message",
        json={
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": question}]}
            ],
            "semantic_view": SEMANTIC_VIEW,
        },
        headers={
            "Authorization": f'Snowflake Token="{token}"',
            "Content-Type": "application/json",
        },
    )
    resp.raise_for_status()
    data = resp.json()

    result_parts = []
    msg = data.get("message") or {}
    for block in msg.get("content", []):
        if block.get("type") == "text":
            result_parts.append(block["text"])
        elif block.get("type") == "sql":
            result_parts.append(f"SQL: {block['statement']}")
            sql_results = session.sql(block["statement"]).collect()
            result_parts.append(json.dumps([row.as_dict() for row in sql_results[:20]], default=str))
        elif block.get("type") == "result_table":
            result_parts.append(json.dumps(block.get("data", [])[:20], default=str))
    return "\n".join(result_parts) if result_parts else "No results returned."


print("Cortex Analyst tool defined.")
test_analytics = query_support_analytics.invoke("How many total support cases are there?")
print(test_analytics[:300])

Cortex Analyst tool defined.
This is our interpretation of your question:

How many total support cases are there over the entire available time period?
SQL: WITH __case_metrics AS (
  SELECT
    case_id,
    case_date
  FROM CUST_SUPPORT_DEMO.SUPPORT.CASE_METRICS
)
SELECT
  MIN(case_date) AS start_date,
  MAX(case_date) AS end


## 3. Build the LangGraph Agent

In [7]:
from langchain_snowflake import ChatSnowflake
from langgraph.prebuilt import create_react_agent

llm = ChatSnowflake(
    session=session,
    model="claude-4-sonnet",
    temperature=0.1,
    max_tokens=2048,
)

SYSTEM_PROMPT = """You are a customer support analytics assistant for a SaaS company.
You have access to two tools:

1. **query_support_analytics** - For quantitative questions about metrics, trends, counts,
   averages, or comparisons. This queries structured data via Cortex Analyst.

2. **search_support_cases** - For questions about specific case details, issue descriptions,
   resolution steps, or searching cases by topic. This searches unstructured case text via Cortex Search.

Be concise and data-driven. When presenting numbers, use appropriate formatting.
If a question involves both metrics and case details, use both tools."""

tools = [search_support_cases, query_support_analytics]

graph = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print(f"LangGraph agent compiled. Nodes: {list(graph.get_graph().nodes.keys())}")

LangGraph agent compiled. Nodes: ['__start__', 'agent', 'tools', '__end__']


## 4. Test the Agent

In [ ]:
response = graph.invoke(
    {"messages": [("user", "What is the average CSAT score by product?")]}
)
print(response["messages"][-1].content)

<Response [200]>
{'message': {'role': 'analyst', 'content': [{'type': 'text', 'text': 'This is our interpretation of your question:\n\nWhat is the average CSAT score by product over the entire available time period?'}, {'type': 'sql', 'statement': 'WITH __case_metrics AS (\n  SELECT\n    product,\n    csat_score\n  FROM CUST_SUPPORT_DEMO.SUPPORT.CASE_METRICS\n)\nSELECT\n  product,\n  AVG(csat_score) AS avg_csat\nFROM __case_metrics\nGROUP BY\n  product\nORDER BY\n  avg_csat DESC NULLS LAST\n -- Generated by Cortex Analyst (request_id: ea5d4b47-2032-484f-87eb-b76ee95c6ef7)\n;', 'confidence': {'verified_query_used': {'name': 'AVG_CSAT_BY_CATEGORY', 'question': 'What is the average CSAT score by issue category?', 'sql': 'SELECT issue_category, AVG(csat_score) AS avg_csat FROM __case_metrics GROUP BY issue_category ORDER BY avg_csat DESC', 'verified_at': 0, 'verified_by': ''}}}]}, 'request_id': 'ea5d4b47-2032-484f-87eb-b76ee95c6ef7', 'warnings': [{'message': "Verified query 'TOTAL_CASES_BY

In [ ]:
response = graph.invoke(
    {"messages": [("user", "Find cases related to authentication failures and how they were resolved")]}
)
print(response["messages"][-1].content)

Based on the search results, I found several authentication failure cases and their resolutions. Here's a summary of the common authentication issues and how they were resolved:

## Authentication Failure Cases & Resolutions

**Primary Issue Type: MFA Token Rejection After Password Reset**

**Problem Description:**
- Customers unable to log in after password reset
- Multi-Factor Authentication (MFA) tokens not being accepted despite correct entry
- Authentication system not recognizing valid MFA codes

**Resolution Steps:**
1. **Reset MFA Configuration** - Clear the existing MFA setup for the affected user
2. **Re-enroll User** - Guide customer through MFA re-enrollment process
3. **Clear Server-Side Cache** - Remove cached authentication tokens from the server
4. **Verify Login** - Confirm successful authentication after re-enrollment

**Root Cause:** The search results indicate this is a recurring pattern where password resets can cause MFA configurations to become desynchronized, le

## 5. Instrument with TruLens + Snowflake AI Observability

We wrap the LangGraph agent with `TruGraph` which:
- Auto-instruments all graph nodes and tool calls
- Captures full execution traces (inputs, outputs, latency)
- Exports traces to Snowflake for the AI Observability UI

In [8]:
from trulens.apps.langgraph import TruGraph
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

APP_NAME = "customer_support_langgraph_agent"
APP_VERSION = "v1_cortex_tools"

tru_graph = TruGraph(
    graph,
    app_name=APP_NAME,
    app_version=APP_VERSION,
    connector=tru_snowflake_connector,
)

print(f"TruGraph registered: {APP_NAME} / {APP_VERSION}")

Running the TruLens dashboard requires providing a `password` to the `SnowflakeConnector`.


✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


/opt/anaconda3/envs/snow_tru_py314/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


instrumenting <class 'langgraph.graph.state.StateGraph'> for base <class 'langgraph.graph.state.StateGraph'>
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.graph.state.CompiledStateGraph'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.pregel.main.Pregel'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
TruGraph registered: customer_support_langgraph_agent / v1_cortex_tools


## 6. Define Evaluation Dataset

We create a test dataset that exercises both tools across different query types.

In [18]:
import pandas as pd

eval_queries = [
    "What is the average CSAT score by product?",
    "Which agent has the fastest average resolution time?",
    "What is the escalation rate for Critical priority cases?",
    "Show me the daily trend of total cases over the last month",
    "How many cases were there for each issue category?",
    "Find cases related to authentication or login failures",
    "What resolutions were used for data loss issues?",
    "Search for cases involving API integration problems",
    "What billing issues have customers reported and how were they resolved?",
    "What is the average first response time by priority level, and show me an example of a Critical case?",
]

queries_df = pd.DataFrame(eval_queries, columns=["query"])

queries_df['query_json'] = queries_df['query'].apply(lambda x: {"messages": [("user", x)]})
print(f"Evaluation dataset: {len(queries_df)} queries")
queries_df

Evaluation dataset: 10 queries


,query,query_json
0,What is the average CSAT score by product?,"{'messages': [('user', 'What is the average CS..."
1,Which agent has the fastest average resolution...,"{'messages': [('user', 'Which agent has the fa..."
2,What is the escalation rate for Critical prior...,"{'messages': [('user', 'What is the escalation..."
3,Show me the daily trend of total cases over th...,"{'messages': [('user', 'Show me the daily tren..."
4,How many cases were there for each issue categ...,"{'messages': [('user', 'How many cases were th..."
5,Find cases related to authentication or login ...,"{'messages': [('user', 'Find cases related to ..."
6,What resolutions were used for data loss issues?,"{'messages': [('user', 'What resolutions were ..."
7,Search for cases involving API integration pro...,"{'messages': [('user', 'Search for cases invol..."
8,What billing issues have customers reported an...,"{'messages': [('user', 'What billing issues ha..."
9,What is the average first response time by pri...,"{'messages': [('user', 'What is the average fi..."


## 7. Execute Evaluation Run

In [19]:
from trulens.core.run import Run
from trulens.core.run import RunConfig
import datetime

run_version = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

run_config = RunConfig(
    run_name=f"batch_run_{run_version}",
    description="Questions for Cortex Analyst",
    dataset_name="SAMPLE_QUERIES_DF",
    source_type="DATAFRAME",
    label="BATCH",
    dataset_spec={
        "RECORD_ROOT.INPUT": "query_json",
    },
    
)

In [21]:
run = tru_graph.add_run(run_config=run_config)

run.start(input_df=queries_df)

## 8. Compute Evaluation Metrics

We compute three key RAG evaluation metrics:
- **Answer Relevance**: Is the response relevant to the user's query?
- **Context Relevance**: Is the retrieved context relevant to the query?
- **Groundedness**: Is the response grounded in the retrieved context?

In [ ]:
import time

metric_list = [
    "answer_relevance",
    "context_relevance",
    "groundedness",
    "coherence",
]

run.compute_metrics(metric_list)
print("Metrics computation started...")

while run.get_status() not in ("COMPLETED", "PARTIALLY_COMPLETED", "CANCELLED"):
    print(f"  Status: {run.get_status()}")
    time.sleep(5)

print(f"Evaluation complete. Final status: {run.get_status()}")

## 9. View Results

Results are available in the **Snowsight AI Observability UI**:

1. Navigate to **AI & ML > Evaluations** in Snowsight
2. Filter by app name: `customer_support_langgraph_agent`
3. Select the run to inspect individual traces and metrics

You can also query the results programmatically:

In [24]:
print(f"Run Name:   {run_config.run_name}")
print(f"App Name:   {APP_NAME}")
print(f"App Version: {APP_VERSION}")
print(f"Final Status: {run.get_status()}")
print()
print("View full results in Snowsight:")
print("  → AI & ML > Evaluations")
print(f"  → Look for app '{APP_NAME}' version '{APP_VERSION}'")

Run Name:   batch_run_2026-03-18 15:28:34
App Name:   customer_support_langgraph_agent
App Version: v1_cortex_tools
Final Status: RunStatus.COMPLETED

View full results in Snowsight:
  → AI & ML > Evaluations
  → Look for app 'customer_support_langgraph_agent' version 'v1_cortex_tools'


In [29]:
eval_df = session.sql(f"""
    SELECT *
    FROM TABLE(
        SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'CUST_SUPPORT_DEMO', 'SUPPORT', '{APP_NAME}', 'EXTERNAL AGENT', '{run_config.run_name}'
        )
    )
    ORDER BY TIMESTAMP DESC
    LIMIT 50
""").to_pandas()

print(f"Evaluation records: {len(eval_df)}")
if not eval_df.empty:
    display(eval_df)
else:
    print("No evaluation data yet. Results may take a moment to appear.")
    print("Check Snowsight → AI & ML → Evaluations for the latest results.")

Evaluation records: 20


,RECORD_ID,INPUT_ID,REQUEST_ID,TIMESTAMP,DURATION_MS,INPUT,OUTPUT,ERROR,GROUND_TRUTH,METRIC_NAME,EVAL_AGG_SCORE,METRIC_TYPE,METRIC_STATUS,METRIC_CALLS,TOTAL_INPUT_TOKENS,TOTAL_OUTPUT_TOKENS,LLM_CALL_COUNT
0,1bfee71e-45d3-4ba4-998a-08a32206954a,obj_hash_1a1ad76671f535770ce0a2de54ae86a8,None,2026-03-18 14:30:46.051345-07:00,10730,What is the average first response time by pri...,## Average First Response Time by Priority Lev...,None,None,answer_relevance,1.000000,answer_relevance,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- The RESPONSE provid...",0,0,0
1,1bfee71e-45d3-4ba4-998a-08a32206954a,obj_hash_1a1ad76671f535770ce0a2de54ae86a8,None,2026-03-18 14:30:46.051345-07:00,10730,What is the average first response time by pri...,## Average First Response Time by Priority Lev...,None,None,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""1. Clarity of data pr...",0,0,0
2,d5ef9150-0613-49df-af19-a863083ac1eb,obj_hash_a290a05a45c335dd0c2cd700a1c8bf6d,None,2026-03-18 14:30:35.319973-07:00,6678,What billing issues have customers reported an...,"Based on the support cases, here are the main ...",None,None,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""Clarity of issue desc...",0,0,0
3,d5ef9150-0613-49df-af19-a863083ac1eb,obj_hash_a290a05a45c335dd0c2cd700a1c8bf6d,None,2026-03-18 14:30:35.319973-07:00,6678,What billing issues have customers reported an...,"Based on the support cases, here are the main ...",None,None,answer_relevance,1.000000,answer_relevance,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- The RESPONSE provid...",0,0,0
4,c63288ab-745b-4990-b8a0-e803cb801a38,obj_hash_90211eb8ad09ba303b2cfae3d6c272af,None,2026-03-18 14:30:28.628593-07:00,6896,Search for cases involving API integration pro...,I found several cases related to API integrati...,None,None,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""Clarity of problem st...",0,0,0
5,c63288ab-745b-4990-b8a0-e803cb801a38,obj_hash_90211eb8ad09ba303b2cfae3d6c272af,None,2026-03-18 14:30:28.628593-07:00,6896,Search for cases involving API integration pro...,I found several cases related to API integrati...,None,None,answer_relevance,1.000000,answer_relevance,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- RESPONSE provides s...",0,0,0
6,cf3deaac-2868-4b02-95fe-e514a4c39ce9,obj_hash_ca8024317c1de9bc9252e269103d0b22,None,2026-03-18 14:30:21.731705-07:00,8892,What resolutions were used for data loss issues?,"Based on the search results, here are the reso...",None,None,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""Clarity of resolution...",0,0,0
7,cf3deaac-2868-4b02-95fe-e514a4c39ce9,obj_hash_ca8024317c1de9bc9252e269103d0b22,None,2026-03-18 14:30:21.731705-07:00,8892,What resolutions were used for data loss issues?,"Based on the search results, here are the reso...",None,None,answer_relevance,1.000000,answer_relevance,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- The RESPONSE provid...",0,0,0
8,1904eadd-dd4b-42ed-b679-4db0b86f5192,obj_hash_5130ff3165ca1a2bf053fa109a6e30c3,None,2026-03-18 14:30:12.837730-07:00,8821,Find cases related to authentication or login ...,"Based on the search results, I found several c...",None,None,answer_relevance,1.000000,answer_relevance,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- The RESPONSE provid...",0,0,0
9,1904eadd-dd4b-42ed-b679-4db0b86f5192,obj_hash_5130ff3165ca1a2bf053fa109a6e30c3,None,2026-03-18 14:30:12.837730-07:00,8821,Find cases related to authentication or login ...,"Based on the search results, I found several c...",None,None,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""Clarity of problem de...",0,0,0


In [32]:
obs_event_sql = f"""SELECT * FROM TABLE(
    SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CUST_SUPPORT_DEMO', 
        'SUPPORT', 
        '{APP_NAME}', 
        'EXTERNAL AGENT')
) 
WHERE RECORD_ATTRIBUTES:"snow.ai.observability.run.name" = '{run_config.run_name}';"""

df = session.sql(obs_event_sql).to_pandas()

df

,TIMESTAMP,START_TIMESTAMP,TRACE,RESOURCE_ATTRIBUTES,RECORD_TYPE,RECORD,RECORD_ATTRIBUTES,VALUE
0,2026-03-18 21:29:09.512121,2026-03-18 21:29:09.511539,"{\n ""span_id"": ""a516a230c43b02cd"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
1,2026-03-18 21:30:25.236816,2026-03-18 21:30:25.235853,"{\n ""span_id"": ""a5d36658105d8a5c"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
2,2026-03-18 21:30:25.233612,2026-03-18 21:30:24.513467,"{\n ""span_id"": ""a93ce47cab503888"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
3,2026-03-18 21:30:25.232383,2026-03-18 21:30:24.531445,"{\n ""span_id"": ""bf96f4af904e18d4"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
4,2026-03-18 21:30:24.780062,2026-03-18 21:30:24.531769,"{\n ""span_id"": ""a7cffdf2d5d3bce6"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
...,...,...,...,...,...,...,...,...
138,2026-03-18 21:30:06.880898,2026-03-18 21:30:04.018130,"{\n ""span_id"": ""8a8770085cf6b8ca"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
139,2026-03-18 21:30:04.019293,2026-03-18 21:30:04.018613,"{\n ""span_id"": ""b9be8cd930ce15e1"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
140,2026-03-18 21:30:04.015833,2026-03-18 21:29:54.151321,"{\n ""span_id"": ""49c108a7b20eb7cc"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None
141,2026-03-18 21:30:04.014884,2026-03-18 21:30:04.014240,"{\n ""span_id"": ""8c813db7c6eb2fc8"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_104a...",None


In [ ]:
df

## Next Steps

- **Iterate on the agent**: Try different LLMs (`llama3.1-70b`, `mistral-large`), adjust the system prompt, or add more tools
- **Compare versions**: Create new app versions and compare evaluation runs side-by-side in Snowsight
- **Add ground truth**: Include expected answers in the dataset to compute the `correctness` metric
- **Add more metrics**: Compute `coherence` or custom metrics for domain-specific evaluation
- **Production deployment**: Use the best-performing configuration to deploy as a Cortex Agent or Streamlit app